In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import librosa
from transformers import WhisperFeatureExtractor, WhisperModel
from tqdm.notebook import tqdm
import joblib
import warnings
warnings.filterwarnings('ignore')

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [2]:
# Define the PyTorch Model Architectures
class TemporalCNNEncoder(nn.Module):
    def __init__(self):
        super(TemporalCNNEncoder, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=512, out_channels=256, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(256)
        self.conv2 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu = nn.GELU()
        self.pool = nn.MaxPool1d(kernel_size=2) 

    def forward(self, x):
        x = x.transpose(1, 2) 
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        return x.transpose(1, 2)

class TransformerBrain(nn.Module):
    def __init__(self, num_classes, d_model=128, seq_length=75):
        super(TransformerBrain, self).__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, seq_length, d_model) * 0.01)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8, dim_feedforward=512, dropout=0.3, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = x + self.pos_embedding
        x = self.transformer(x)
        x = x.mean(dim=1) 
        return self.fc(x)

class EndToEndStutterModel(nn.Module):
    def __init__(self, num_classes):
        super(EndToEndStutterModel, self).__init__()
        self.cnn = TemporalCNNEncoder()
        self.transformer = TransformerBrain(num_classes)

    def forward(self, x):
        return self.transformer(self.cnn(x))

In [3]:
# Load Uclass Labels
df_labels = pd.read_csv('../ksh/stl/uclass_sep28k-style_labels.csv')
# The Clip names are already fully formed in this CSV, so no need to construct them!

# Extract Whisper Embeddings
print("Loading Whisper...")
model_name = "openai/whisper-base"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)
whisper_model = WhisperModel.from_pretrained(model_name).to(device)

def get_whisper_embedding_sequence(audio_path):
    if not os.path.exists(audio_path): 
        return None
    try:
        speech_array, sampling_rate = librosa.load(audio_path, sr=16000)
        inputs = feature_extractor(speech_array, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)
        with torch.no_grad():
            encoder_outputs = whisper_model.encoder(input_features)
        return encoder_outputs.last_hidden_state.squeeze(0).cpu().numpy()
    except Exception as e:
        return None

print("Extracting embeddings...")
valid_clips = []
embeddings = []

for _, row in tqdm(df_labels.iterrows(), total=len(df_labels)):
    clip_name = row['Clip']
    audio_path = f"Uclass-clips/clips/{clip_name}.wav"
    
    emb = get_whisper_embedding_sequence(audio_path)
    if emb is not None:
        valid_clips.append(clip_name)
        # Slice to 150 tokens as done during training
        embeddings.append(emb[:150, :])

X = np.array(embeddings)
print(f"Extracted embeddings shape: {X.shape}")
X_tensor = torch.tensor(X, dtype=torch.float32).to(device)

Loading Whisper...


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Extracting embeddings...


  0%|          | 0/4712 [00:00<?, ?it/s]

Extracted embeddings shape: (4712, 150, 512)


In [4]:
# Evaluate Models
predictions = {'Clip': valid_clips}

model_names = ['block', 'prolongation', 'SoundRep', 'cnn_transformer']
# Map to CSV columns if they match
col_map = {
    'block': 'Block',
    'prolongation': 'Prolongation',
    'SoundRep': 'SoundRep',
    'cnn_transformer': 'Any'
}

for m_name in model_names:
    try:
        le_path = f"models/{m_name}_label_encoder.pkl"
        model_path = f"models/{m_name}_model.pt"
        
        if not os.path.exists(le_path) or not os.path.exists(model_path):
            print(f"Skipping {m_name}, files not found.")
            continue
            
        le = joblib.load(le_path)
        num_classes = len(le.classes_)
        
        model = EndToEndStutterModel(num_classes).to(device)
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()
        
        with torch.no_grad():
            outputs = model(X_tensor)
            _, predicted = torch.max(outputs.data, 1)
            predicted_classes = le.inverse_transform(predicted.cpu().numpy())
            
            # Map predictions
            binary_preds = []
            for p in predicted_classes:
                if str(p).lower() in ['0', 'fluent', 'nostutteredwords', 'naturalpause', 'music', 'nospeech']:
                    binary_preds.append(0)
                else:
                    binary_preds.append(1)
            
            col_name = col_map.get(m_name, m_name)
            predictions[col_name] = binary_preds
            print(f"Successfully ran inference for {m_name} -> {col_name}")
            
    except Exception as e:
        print(f"Error evaluating {m_name}: {e}")

df_preds = pd.DataFrame(predictions)
df_preds.to_csv('predictions_uclass.csv', index=False)
print("Saved predictions to predictions_uclass.csv")

Successfully ran inference for block -> Block
Successfully ran inference for prolongation -> Prolongation
Successfully ran inference for SoundRep -> SoundRep
Successfully ran inference for cnn_transformer -> Any
Saved predictions to predictions_uclass.csv


In [5]:
# Run the evaluation script
# Note: Because the Uclass labels are 0 or 1, we set --reference_threshold 1
!python ../archive/Testing/evaluate.py --reference ../ksh/stl/uclass_sep28k-style_labels.csv --prediction predictions_uclass.csv --reference_threshold 1


# Any
## tp = 1390 , tn = 1561 , fp = 661 , fn = 1100
## precision = 0.678 , recall = 0.558 , f1 = 0.612 , accuracy = 0.626 , specificity = 0.703

# Block
## tp = 934 , tn = 2347 , fp = 940 , fn = 491
## precision = 0.498 , recall = 0.655 , f1 = 0.566 , accuracy = 0.696 , specificity = 0.714

# Prolongation
## tp = 314 , tn = 2865 , fp = 1404 , fn = 129
## precision = 0.183 , recall = 0.709 , f1 = 0.291 , accuracy = 0.675 , specificity = 0.671

# SoundRep
## tp = 913 , tn = 2245 , fp = 817 , fn = 737
## precision = 0.528 , recall = 0.553 , f1 = 0.54 , accuracy = 0.67 , specificity = 0.733
